# 00 - Dataset Ingestion and Preparation

This notebook manages the initial setup, ingestion, validation, and export of the **Corr2Cause** dataset for the causal NLP reasoning pipeline.

### Sections
1. **Project Setup**: Initialize directory paths and configure reusable export and validation logic.
2. **Load Corr2Cause Dataset**: Validate canonical local datasets or ingest the benchmark from Hugging Face Hub.
3. **Inspect Dataset Splits**: Examine split record counts and partition structure.
4. **Inspect Dataset Schema and Samples**: Analyze feature definitions, missing values, duplicates, and sample records.
5. **Analyze Label Distribution**: Evaluate class balance, variable counts, and causal template frequencies.
6. **Export Original Splits**: Perform preflight conflict checks and transactional export/reuse of raw splits.
7. **Combine Training and Validation**: Concatenate training and validation sets for downstream training.
8. **Export Combined Dataset**: Export/reuse combined data and separate test partition in `data/processed/`.
9. **Verify Exported Datasets**: Audit all exported CSV files for schema, row counts, and data integrity across all columns.
10. **Summary**: Synthesize dataset characteristics, split isolation, and pipeline next steps.

## 1. Project Setup

Initialize repository-relative directory paths for raw and processed datasets, define target export paths, and configure reusable data verification and safe export routines.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
from datasets import load_dataset

def find_repo_root() -> Path:
    """Locate the repository root whether run from root, AML/, DL/, or subfolders."""
    current = Path.cwd().resolve()
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "00_setup.ipynb").exists() and (candidate / "data").exists():
            return candidate
        if (candidate / ".git").exists() and (candidate / "AML").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate repository root from cwd: '{current}'. "
        "Ensure notebook resides within the causal-nlp-reasoning repository."
    )

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Standardized output file paths across pipeline
EXPECTED_EXPORT_PATHS = {
    "raw_train": RAW_DATA_DIR / "train.csv",
    "raw_val": RAW_DATA_DIR / "validation.csv",
    "raw_test": RAW_DATA_DIR / "test.csv",
    "processed_train_val": PROCESSED_DATA_DIR / "train_val.csv",
    "processed_test": PROCESSED_DATA_DIR / "test.csv",
}

# -----------------------------------------------------------------------------
# Configuration: Normal Setup vs. Fresh Rebuild Workflow
# -----------------------------------------------------------------------------
# FORCE_REBUILD:
#   False (normal setup): Safe, non-destructive mode. Reuses valid, pre-existing
#          exported files without rewriting.
#   True  (fresh data rebuild): Explicitly regenerates the five canonical CSV
#          files from Hugging Face: data/raw/{train,validation,test}.csv and
#          data/processed/{train_val,test}.csv. Never touches other directories.
FORCE_REBUILD = False

print(f"Project root             : {REPO_ROOT.name}")
print(f"Raw data directory       : {RAW_DATA_DIR.relative_to(REPO_ROOT)}")
print(f"Processed data directory : {PROCESSED_DATA_DIR.relative_to(REPO_ROOT)}")
print(f"Rebuild workflow mode    : {'FRESH REBUILD (FORCE_REBUILD=True)' if FORCE_REBUILD else 'SAFE REUSE'}")
print("Configured 5 expected destinations with safe export and verification routines.")


Project root             : causal-nlp-reasoning
Raw data directory       : data/raw
Processed data directory : data/processed
Rebuild workflow mode    : SAFE REUSE
Configured 5 expected destinations with safe export and verification routines.


### Safe Export and Verification Helpers
Helper routines to verify schema fidelity and prevent accidental overwrites.

In [2]:
def verify_dataframe_match(df_loaded, df_expected, dataset_label=""):
    """Verify df_loaded matches df_expected across columns, count, and values.
    Returns (True, None) if match, or (False, reason_str) if mismatch.
    """
    if len(df_loaded) != len(df_expected):
        return False, f"Row count mismatch: found {len(df_loaded):,}, expected {len(df_expected):,}"
    
    if list(df_loaded.columns) != list(df_expected.columns):
        return False, f"Column mismatch: found {list(df_loaded.columns)}, expected {list(df_expected.columns)}"
    
    for col in df_expected.columns:
        s_expected = df_expected[col].fillna("").astype(str).reset_index(drop=True)
        s_loaded = df_loaded[col].fillna("").astype(str).reset_index(drop=True)
        if not s_loaded.equals(s_expected):
            diff_indices = (s_loaded != s_expected)
            first_idx = int(diff_indices.idxmax())
            return False, (
                f"Value mismatch in column '{col}' at record index {first_idx}: "
                f"found '{s_loaded[first_idx][:40]}...', expected '{s_expected[first_idx][:40]}...'"
            )
            
    return True, None


### Preflight Verification Across Destinations
Inspect destination paths prior to export.


In [3]:
def preflight_check_all_destinations(destinations_dict, force_rebuild=False):
    """Inspect all expected output destinations before any writing takes place.
    If force_rebuild is False and any existing file contains conflicting data, halts
    before any missing file is written.
    """
    conflicts = []
    preflight_records = []
    for name, (path, df_expected) in destinations_dict.items():
        rel_path = str(path.relative_to(REPO_ROOT))
        if path.exists():
            df_existing = pd.read_csv(path)
            is_match, reason = verify_dataframe_match(df_existing, df_expected, dataset_label=name)
            if force_rebuild:
                status = "Will overwrite (Fresh Rebuild)"
                preflight_records.append({"Dataset": name, "Path": rel_path, "Status": status, "Rows": len(df_existing), "Detail": "Target marked for fresh export"})
            elif is_match:
                status = "Already exists and verified"
                preflight_records.append({"Dataset": name, "Path": rel_path, "Status": status, "Rows": len(df_existing), "Detail": "Matches expected dataset"})
            else:
                status = "Conflict detected"
                conflicts.append((rel_path, reason))
                preflight_records.append({"Dataset": name, "Path": rel_path, "Status": status, "Rows": len(df_existing), "Detail": reason})
        else:
            status = "Missing (ready to export)"
            preflight_records.append({"Dataset": name, "Path": rel_path, "Status": status, "Rows": len(df_expected), "Detail": "File does not exist yet"})

    if conflicts and not force_rebuild:
        conflict_msg = "\n".join(f"  - {p}: {err}" for p, err in conflicts)
        raise ValueError(
            f"Preflight validation failed: Existing file(s) contain conflicting data:\n{conflict_msg}\n"
            "Execution halted before writing any files. Conflicting files are preserved without modification."
        )
    return pd.DataFrame(preflight_records)


### Safe Export, Staging Handler, and Transactional Rebuild Protocol
Provides atomic file replacements, idempotent exports, and coordinated multi-file transactional rebuilds with staging, full-fidelity chunked pre-commit validation, and verified rollback recovery.

**Safety & Atomicity Guarantees**:
- **Atomic Single-File Writes**: Writes to a temporary file on the same filesystem and uses `os.replace()`. Canonical destination files are never unlinked before write completion and verification.
- **Startup Protection**: Never automatically deletes `.rebuild_backup` at startup; halts with clear recovery instructions if existing backup or recovery data is detected to protect data from interrupted runs.
- **Transactional Staging & Pre-commit Validation**: When replacing canonical datasets (`FORCE_REBUILD=True` or initial export), all five datasets are written to a staging directory (`data/.rebuild_staging`), thoroughly validated before any canonical file is touched, backed up to `data/.rebuild_backup`, and swapped via `os.replace()`.
- **Pre-transaction Inventory & Asymmetric Rollback**: Tracks which destinations existed before the transaction; on failure, restores every pre-existing file from backup and removes every file newly created by the failed transaction.
- **Unswallowed Rollback Exceptions & Verification**: Captures all rollback errors, verifies file existence and SHA-256 hashes, and retains backups whenever rollback is incomplete or uncertain.
- **Conditional Backup Cleanup**: Deletes backups only after a successful rebuild or a verified rollback. Ensures cleanup failures cannot destroy the only recoverable copy.
- **Filesystem & Crash-Recovery Boundaries**: While `os.replace()` provides POSIX-level atomic replacement for individual files, POSIX does not natively provide multi-file transactional commits across separate directory entries. In the rare event of an unrecoverable hardware power cut during the replacement loop, the preserved backup directory (`data/.rebuild_backup`) provides a deterministic manual recovery point.

In [4]:
def export_dataset(path, df_expected, dataset_name="Dataset", force_rebuild=False):
    """Safely exports missing datasets, reuses verified existing datasets, or
    atomically replaces canonical files when force_rebuild=True.
    
    Safety constraints:
    - Never deletes existing files before writing and validating replacements.
    - Writes to an isolated temporary file on the same filesystem first.
    - Re-reads and strictly verifies exported file contents before atomic replacement via os.replace().
    """
    path = Path(path)
    rel_path = str(path.relative_to(REPO_ROOT))
    allowed_targets = set(EXPECTED_EXPORT_PATHS.values())

    if path.exists() and not force_rebuild:
        df_loaded = pd.read_csv(path)
        is_match, reason = verify_dataframe_match(df_loaded, df_expected, dataset_label=dataset_name)
        if is_match:
            return {
                "Dataset": dataset_name,
                "Path": rel_path,
                "Rows": len(df_loaded),
                "Action": "Verified and Reused (Matches existing)",
            }
        else:
            raise ValueError(
                f"Conflict detected for '{dataset_name}' at {rel_path}: {reason}. "
                "Will not overwrite existing file."
            )

    if force_rebuild and path not in allowed_targets:
        raise PermissionError(f"Safety restriction: Refusing to target non-canonical path: {rel_path}")

    path.parent.mkdir(parents=True, exist_ok=True)
    
    # Atomic write pattern: write to temporary sibling file, verify, and os.replace()
    temp_path = path.parent / f".tmp_{path.name}.{os.getpid()}"
    try:
        df_expected.to_csv(temp_path, index=False)
        df_written = pd.read_csv(temp_path)
        is_match, reason = verify_dataframe_match(df_written, df_expected, dataset_label=dataset_name)
        if not is_match:
            raise ValueError(f"Post-write validation failed for {rel_path}: {reason}")
        os.replace(temp_path, path)
    finally:
        if temp_path.exists():
            temp_path.unlink()

    return {
        "Dataset": dataset_name,
        "Path": rel_path,
        "Rows": len(df_expected),
        "Action": "Exported fresh dataset and verified",
    }

# Backward compatibility alias
export_or_reuse_dataset = export_dataset


def verify_processed_train_val_consistency(raw_train_path, raw_val_path, proc_train_val_path, chunk_size=50000):
    """Verify that processed_train_val identically matches raw_train followed by raw_val
    across all records and all columns ('input', 'label', 'num_variables', 'template')
    using memory-efficient chunked streaming.
    
    Returns (True, None) if match, or (False, reason_str) if mismatch.
    """
    proc_reader = pd.read_csv(proc_train_val_path, chunksize=chunk_size)
    buffer = pd.DataFrame()

    def get_proc_rows(needed_count):
        nonlocal buffer, proc_reader
        collected = [buffer] if not buffer.empty else []
        tot = len(buffer)
        while tot < needed_count:
            try:
                chunk = next(proc_reader)
                collected.append(chunk)
                tot += len(chunk)
            except StopIteration:
                break
        if not collected:
            return pd.DataFrame()
        full_df = pd.concat(collected, ignore_index=True)
        res = full_df.iloc[:needed_count].reset_index(drop=True)
        buffer = full_df.iloc[needed_count:].reset_index(drop=True)
        return res

    record_idx = 0
    # 1. Verify train partition across all columns
    for train_chunk in pd.read_csv(raw_train_path, chunksize=chunk_size):
        n = len(train_chunk)
        proc_sub = get_proc_rows(n)
        if len(proc_sub) != n:
            return False, f"Processed row deficit in train partition: expected {n}, got {len(proc_sub)}"
        
        for col in train_chunk.columns:
            s_raw = train_chunk[col].fillna("").astype(str).reset_index(drop=True)
            s_proc = proc_sub[col].fillna("").astype(str).reset_index(drop=True)
            if not s_raw.equals(s_proc):
                diff_mask = (s_raw != s_proc)
                rel_idx = int(diff_mask.idxmax())
                abs_idx = record_idx + rel_idx
                return False, (
                    f"Value mismatch in train partition column '{col}' at row {abs_idx}: "
                    f"raw='{s_raw[rel_idx][:40]}...', proc='{s_proc[rel_idx][:40]}...'"
                )
        record_idx += n

    # 2. Verify validation partition across all columns
    raw_val_df = pd.read_csv(raw_val_path)
    n_val = len(raw_val_df)
    proc_val_sub = get_proc_rows(n_val)
    if len(proc_val_sub) != n_val:
        return False, f"Processed row deficit in val partition: expected {n_val}, got {len(proc_val_sub)}"

    for col in raw_val_df.columns:
        s_raw = raw_val_df[col].fillna("").astype(str).reset_index(drop=True)
        s_proc = proc_val_sub[col].fillna("").astype(str).reset_index(drop=True)
        if not s_raw.equals(s_proc):
            diff_mask = (s_raw != s_proc)
            rel_idx = int(diff_mask.idxmax())
            abs_idx = record_idx + rel_idx
            return False, (
                f"Value mismatch in validation partition column '{col}' at row {abs_idx}: "
                f"raw='{s_raw[rel_idx][:40]}...', proc='{s_proc[rel_idx][:40]}...'"
            )
    record_idx += n_val

    # 3. Check for leftover rows
    trailing = get_proc_rows(1)
    if not trailing.empty:
        return False, "Processed train_val contains unexpected trailing rows beyond train and validation sets"

    return True, None


def validate_canonical_local_datasets(export_paths=EXPECTED_EXPORT_PATHS, expected_counts=None, expected_columns=None):
    """Validate existence, CSV structure, expected columns, row counts, and cross-split
    consistency for all five canonical export files without redundant memory-intensive reloading.

    Safety & Performance Guarantees:
    - Never assumes validity based solely on file existence or file size.
    - Accurately categorizes missing files vs. existing files that fail validation.
    - Validates CSV parsability, column names, and column order using sample scans.
    - Accurately counts rows via single-column scan (`usecols=['label']`), verifying
      complete CSV parsing integrity without repeatedly loading 360MB text into memory.
    - Validates exact parity between raw test and processed test using `verify_dataframe_match`.
    - Validates processed train_val partition alignment (record count equality, complete chunked
      content comparison across all columns including 'input' and 'template', and absence of trailing rows).

    Returns:
        dict:
            "can_reuse": bool,
            "all_exist": bool,
            "missing_files": list of str,
            "invalid_files": list of dict {"path": str, "reason": str},
            "validation_details": list of dict,
            "summary_reason": str or None
    """
    if expected_counts is None:
        expected_counts = {
            "raw_train": 205734,
            "raw_val": 1076,
            "raw_test": 1162,
            "processed_train_val": 206810,
            "processed_test": 1162,
        }
    if expected_columns is None:
        expected_columns = ["input", "label", "num_variables", "template"]

    details = []
    missing_files = []
    invalid_files = []

    # 1. Existence check for all 5 canonical files
    for key, path in export_paths.items():
        try:
            rel = str(path.relative_to(REPO_ROOT))
        except ValueError:
            rel = str(path)
        if not path.exists():
            missing_files.append(rel)

    if missing_files:
        return {
            "can_reuse": False,
            "all_exist": False,
            "missing_files": missing_files,
            "invalid_files": [],
            "validation_details": [],
            "summary_reason": f"Missing canonical file(s): {', '.join(missing_files)}",
        }

    # 2. CSV parsability and column schema check
    for key, path in export_paths.items():
        try:
            rel = str(path.relative_to(REPO_ROOT))
        except ValueError:
            rel = str(path)
        try:
            sample_df = pd.read_csv(path, nrows=5)
        except Exception as e:
            invalid_files.append({"path": rel, "reason": f"CSV parsing error: {e}"})
            continue

        if list(sample_df.columns) != list(expected_columns):
            invalid_files.append({
                "path": rel,
                "reason": f"Column mismatch: expected {expected_columns}, got {list(sample_df.columns)}"
            })

    if invalid_files:
        return {
            "can_reuse": False,
            "all_exist": True,
            "missing_files": [],
            "invalid_files": invalid_files,
            "validation_details": [],
            "summary_reason": f"Schema or CSV format failure in {len(invalid_files)} file(s)",
        }

    # 3. Row count validation (lightweight column scan validates every row in file)
    row_counts = {}
    for key, path in export_paths.items():
        try:
            rel = str(path.relative_to(REPO_ROOT))
        except ValueError:
            rel = str(path)
        try:
            col_df = pd.read_csv(path, usecols=["label"])
            actual_count = len(col_df)
            row_counts[key] = actual_count
        except Exception as e:
            invalid_files.append({"path": rel, "reason": f"Failed reading row count: {e}"})
            continue

        expected = expected_counts[key]
        if actual_count != expected:
            invalid_files.append({
                "path": rel,
                "reason": f"Row count mismatch: expected {expected:,}, got {actual_count:,}"
            })
        else:
            details.append({
                "Key": key,
                "Path": rel,
                "Rows": actual_count,
                "Columns": len(expected_columns),
                "Status": "Valid structure & count",
            })

    if invalid_files:
        return {
            "can_reuse": False,
            "all_exist": True,
            "missing_files": [],
            "invalid_files": invalid_files,
            "validation_details": details,
            "summary_reason": f"Row count mismatch in {len(invalid_files)} file(s)",
        }

    # 4. Consistency checks between raw and processed splits
    try:
        df_raw_test = pd.read_csv(export_paths["raw_test"])
        df_proc_test = pd.read_csv(export_paths["processed_test"])
        is_match, reason = verify_dataframe_match(df_proc_test, df_raw_test, "Processed vs Raw Test")
        if not is_match:
            invalid_files.append({"path": str(export_paths["processed_test"]), "reason": f"Test parity mismatch: {reason}"})
    except Exception as e:
        invalid_files.append({"path": str(export_paths["processed_test"]), "reason": f"Error validating test parity: {e}"})

    try:
        is_match, reason = verify_processed_train_val_consistency(
            export_paths["raw_train"],
            export_paths["raw_val"],
            export_paths["processed_train_val"],
            chunk_size=50000,
        )
        if not is_match:
            invalid_files.append({
                "path": str(export_paths["processed_train_val"]),
                "reason": f"Full chunked content mismatch: {reason}"
            })
    except Exception as e:
        invalid_files.append({
            "path": str(export_paths["processed_train_val"]),
            "reason": f"Error running chunked train_val verification: {e}"
        })

    if invalid_files:
        return {
            "can_reuse": False,
            "all_exist": True,
            "missing_files": [],
            "invalid_files": invalid_files,
            "validation_details": details,
            "summary_reason": f"Cross-split parity or content consistency failed in {len(invalid_files)} file(s)",
        }

    return {
        "can_reuse": True,
        "all_exist": True,
        "missing_files": [],
        "invalid_files": [],
        "validation_details": details,
        "summary_reason": None,
    }


def compute_file_sha256(file_path):
    """Compute SHA-256 hex digest for a file in 64KB blocks."""
    import hashlib
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()


def rebuild_all_canonical_datasets_transactional(destinations_dict, base_data_dir=None, expected_counts=None, expected_columns=None):
    """Safely rebuild and replace canonical datasets using a staged transaction strategy with verified rollback.

    Transaction Safety & Rollback Architecture:
    1. Startup Recovery Protection: Never automatically deletes an existing `.rebuild_backup` directory.
       If `.rebuild_backup` is detected at startup, execution halts with explicit recovery instructions
       to preserve recovery data from interrupted runs.
    2. Pre-transaction Inventory: Tracks which canonical destinations existed before the transaction
       and records their SHA-256 content hashes.
    3. Pre-commit Staging & Validation: Stages all replacement datasets in `.rebuild_staging` on the
       same filesystem and runs strict schema, count, and chunked cross-split content checks before
       touching any canonical files.
    4. Non-Destructive Backup: Copies existing canonical files to `.rebuild_backup` and verifies SHA-256
       hashes before beginning any replacements.
    5. Coordinated Replacement & Rollback:
       - Replaces canonical files using atomic `os.replace()`.
       - If any replacement step fails, rollback restores every previously existing file from backup
         (via copy to a hidden temporary restore file and atomic replace, preserving the backup copy)
         and unlinks every file newly created by the failed transaction.
       - Rollback exceptions are never swallowed.
       - Verification checks that all pre-existing files exist with intact hashes and that all newly
         created files are removed.
       - Backups are retained on disk whenever rollback is incomplete or uncertain.
       - Backups are deleted only after a successful rebuild or a verified rollback. Cleanup failures
         cannot destroy the only recoverable copy.

    Note: POSIX provides atomic replacement for individual files via os.replace(), but does not
    provide native multi-file filesystem transactions across multiple directories.
    """
    import os
    import shutil
    import warnings
    from pathlib import Path

    if base_data_dir is None:
        base_data_dir = REPO_ROOT / "data"
    base_data_dir = Path(base_data_dir)
    staging_dir = base_data_dir / ".rebuild_staging"
    backup_dir = base_data_dir / ".rebuild_backup"

    # 1. Startup Protection: Never delete an existing .rebuild_backup directory at startup.
    # Detect and preserve recovery data from interrupted runs; stop with clear recovery instructions.
    if backup_dir.exists():
        backup_files = [p for p in backup_dir.rglob("*") if p.is_file()]
        raise RuntimeError(
            f"Existing backup directory detected at '{backup_dir}' ({len(backup_files)} recovery file(s) found).\n"
            "A previous rebuild run may have been interrupted or terminated before completing.\n"
            "To preserve potential recovery data, automatic startup deletion is prohibited and execution is halted.\n"
            "Recovery instructions:\n"
            f"1. Check the integrity of canonical datasets in '{base_data_dir}'.\n"
            f"2. Inspect '{backup_dir}' and manually restore any needed files.\n"
            f"3. Once canonical datasets are verified or restored, manually remove '{backup_dir}'.\n"
            "4. Re-run the rebuild process."
        )

    # Reset staging directory (uncommitted temporary files only)
    shutil.rmtree(staging_dir, ignore_errors=True)

    # 2. Track which canonical destinations existed before the transaction
    pre_existing = {}
    pre_existing_hashes = {}
    for key, (canonical_path, df, label) in destinations_dict.items():
        c_path = Path(canonical_path)
        exists = c_path.exists()
        pre_existing[key] = exists
        if exists:
            pre_existing_hashes[key] = compute_file_sha256(c_path)

    staging_paths = {}
    backed_up_files = {}

    try:
        # 3. Stage all replacements in staging directory on the same filesystem
        for key, (canonical_path, df, label) in destinations_dict.items():
            c_path = Path(canonical_path)
            try:
                rel = c_path.relative_to(base_data_dir)
            except ValueError:
                rel = c_path.name
            staged_path = staging_dir / rel
            staged_path.parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(staged_path, index=False)
            staging_paths[key] = staged_path

        # Pre-commit validation of all staged files before modifying any canonical files
        validation = validate_canonical_local_datasets(
            staging_paths,
            expected_counts=expected_counts,
            expected_columns=expected_columns,
        )
        if not validation["can_reuse"]:
            raise ValueError(f"Staged dataset pre-commit validation failed: {validation['summary_reason']}")

        # 4. Backup all pre-existing canonical files
        if any(pre_existing.values()):
            backup_dir.mkdir(parents=True, exist_ok=True)
            for key, (canonical_path, df, label) in destinations_dict.items():
                if pre_existing[key]:
                    c_path = Path(canonical_path)
                    try:
                        rel = c_path.relative_to(base_data_dir)
                    except ValueError:
                        rel = c_path.name
                    b_file = backup_dir / rel
                    b_file.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(c_path, b_file)
                    b_hash = compute_file_sha256(b_file)
                    if b_hash != pre_existing_hashes[key]:
                        raise IOError(f"Backup copy verification failed for '{c_path}': hash mismatch")
                    backed_up_files[key] = b_file

        # 5. Atomic replacement loop with rollback
        replace_error = None
        try:
            for key, (canonical_path, df, label) in destinations_dict.items():
                c_path = Path(canonical_path)
                staged_path = staging_paths[key]
                c_path.parent.mkdir(parents=True, exist_ok=True)
                os.replace(staged_path, canonical_path)
        except Exception as exc:
            replace_error = exc

        if replace_error is not None:
            # ROLLBACK
            rollback_errors = []

            # 5a. Restore every previously existing file from backup
            for key, (canonical_path, df, label) in destinations_dict.items():
                c_path = Path(canonical_path)
                if pre_existing[key]:
                    b_file = backed_up_files.get(key)
                    if b_file is None or not b_file.exists():
                        rollback_errors.append(f"Backup file missing for '{c_path}' during rollback")
                        continue
                    tmp_restore = c_path.with_name(f".{c_path.name}.restore_tmp")
                    try:
                        c_path.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(b_file, tmp_restore)
                        os.replace(tmp_restore, c_path)
                    except Exception as r_err:
                        if tmp_restore.exists():
                            try:
                                tmp_restore.unlink()
                            except Exception:
                                pass
                        rollback_errors.append(f"Failed restoring '{c_path}': {r_err}")

            # 5b. Remove files newly created by the failed transaction
            for key, (canonical_path, df, label) in destinations_dict.items():
                c_path = Path(canonical_path)
                if not pre_existing[key]:
                    if c_path.exists():
                        try:
                            c_path.unlink()
                        except Exception as u_err:
                            rollback_errors.append(f"Failed removing newly created file '{c_path}': {u_err}")

            # 5c. Verify restoration and file states
            verification_failures = []
            for key, (canonical_path, df, label) in destinations_dict.items():
                c_path = Path(canonical_path)
                if pre_existing[key]:
                    if not c_path.exists():
                        verification_failures.append(f"Restoration verification failed: '{c_path}' does not exist")
                    else:
                        curr_hash = compute_file_sha256(c_path)
                        if curr_hash != pre_existing_hashes[key]:
                            verification_failures.append(
                                f"Restoration verification failed: '{c_path}' hash mismatch "
                                f"(expected {pre_existing_hashes[key][:8]}, got {curr_hash[:8]})"
                            )
                else:
                    if c_path.exists():
                        verification_failures.append(
                            f"Cleanup verification failed: newly created file '{c_path}' still exists"
                        )

            if not rollback_errors and not verification_failures:
                # Verified rollback: delete backup directory
                if backup_dir.exists():
                    try:
                        shutil.rmtree(backup_dir)
                    except Exception as cleanup_err:
                        warnings.warn(f"Failed to remove backup directory '{backup_dir}' after verified rollback: {cleanup_err}")
                raise RuntimeError(
                    f"Transaction failed during replacement: {replace_error}. "
                    "Rollback successfully verified: all pre-existing files restored and all newly created files removed."
                ) from replace_error
            else:
                # Rollback incomplete or uncertain: RETAIN BACKUPS!
                failures_summary = "; ".join(rollback_errors + verification_failures)
                raise RuntimeError(
                    f"Transaction failed during replacement: {replace_error}. "
                    f"CRITICAL: Rollback failed or was incomplete ({failures_summary}). "
                    f"Recovery backups are PRESERVED in '{backup_dir}' for manual inspection and recovery."
                ) from replace_error

        # Success: delete backup directory only after successful rebuild
        if backup_dir.exists():
            try:
                shutil.rmtree(backup_dir)
            except Exception as cleanup_err:
                warnings.warn(f"Failed to remove backup directory '{backup_dir}' after successful rebuild: {cleanup_err}")

        reports = []
        for key, (canonical_path, df, label) in destinations_dict.items():
            c_path = Path(canonical_path)
            try:
                rel = str(c_path.relative_to(REPO_ROOT))
            except ValueError:
                rel = str(c_path)
            reports.append({
                "Dataset": label,
                "Path": rel,
                "Rows": len(df),
                "Action": "Rebuilt fresh dataset via transactional staging & verified",
            })
        return reports
    finally:
        # Staging directory is temporary and safe to clean up
        shutil.rmtree(staging_dir, ignore_errors=True)
        # Ensure cleanup failures cannot destroy the only recoverable copy:
        # backup_dir is NEVER automatically deleted in finally.
        # It is ONLY deleted after a successful rebuild or a verified complete rollback.

## 2. Load Corr2Cause Dataset

The **Corr2Cause** benchmark (Jin et al., 2023) evaluates natural language processing models on causal reasoning from correlational statements.

The task is framed as natural language inference:
- **Premise**: Textual description of statistical relations (correlations and conditional independencies) in a closed system of variables.
- **Hypothesis**: A candidate causal relationship between specific variables.
- **Label**: Binary target indicating whether the causal hypothesis is logically entailed (`1`) or refuted (`0`) by the premise.

### Ingestion, Integrity Validation, and Transactional Rebuild Protocol
To prevent redundant Hugging Face downloads while strictly guarding against data loss, silent corruption, or mixed-state writes:
1. **Pre-check Full-Fidelity Validation**: Before accessing Hugging Face Hub, the notebook validates all five canonical export targets (`data/raw/{train,validation,test}.csv` and `data/processed/{train_val,test}.csv`). Validation verifies file existence, CSV syntax, column schema, exact row counts, and chunked value-level parity across all columns (including `input` and `template`).
2. **Safe Reuse Mode (`FORCE_REBUILD = False` & All Valid)**: If all five files exist and pass validation, Hugging Face Hub loading and export operations are safely skipped (`dataset = None`). Existing files are reused without re-writing disk storage.
3. **Invalid Existing File Guard (`FORCE_REBUILD = False` & Validation Failure)**: If any canonical file already exists on disk but is corrupt, truncated, or conflicting, the pipeline **halts immediately** with explicit recovery instructions. It will **never silently overwrite** an existing conflicting file.
4. **Initial Setup Mode (`FORCE_REBUILD = False` & Files Missing)**: If canonical files are not present on disk (e.g. first-time repository clone), the pipeline automatically connects to Hugging Face Hub, ingests `causal-nlp/corr2cause`, and exports the canonical datasets via transactional staging.
5. **Fresh Rebuild Mode (`FORCE_REBUILD = True`)**: When explicitly requested by setting `FORCE_REBUILD = True` in Section 1, the pipeline executes an intentional, transaction-safe rebuild: it downloads the benchmark, stages all five datasets in `data/.rebuild_staging`, validates every file before touching canonical data, backs up existing files to `data/.rebuild_backup`, and atomically swaps them via `os.replace()` with automated rollback.

In [5]:
DATASET_NAME = "causal-nlp/corr2cause"

# Validate canonical local files before making network requests or loading datasets
validation_report = validate_canonical_local_datasets(EXPECTED_EXPORT_PATHS)

if validation_report["can_reuse"] and not FORCE_REBUILD:
    print("✓ All 5 canonical local CSV files exist and passed full chunked content validation.")
    print("✓ Skipping Hugging Face Hub dataset load and export operations (Safe Reuse Mode).")
    df_validation_summary = pd.DataFrame(validation_report["validation_details"])
    display(df_validation_summary)
    dataset = None
elif FORCE_REBUILD:
    print("FORCE_REBUILD is enabled: executing explicitly authorized fresh rebuild from Hugging Face Hub.")
    print(f"Loading '{DATASET_NAME}' from Hugging Face Hub...")
    dataset = load_dataset(DATASET_NAME)
    print("Dataset successfully loaded from Hugging Face Hub.")
    print(dataset)
else:
    # FORCE_REBUILD is False, but validation failed for reuse
    if validation_report["invalid_files"]:
        invalid_summary = "\n".join(
            f"  • {item['path']}: {item['reason']}" for item in validation_report["invalid_files"]
        )
        raise ValueError(
            f"Preflight validation failed for existing canonical dataset file(s):\n{invalid_summary}\n\n"
            "Safety Guard Activated: One or more canonical files already exist on disk but contain corrupted, "
            "incomplete, or conflicting data. To protect local data from unintended loss or silent overwriting, "
            "automatic overwriting is prohibited when FORCE_REBUILD=False.\n\n"
            "Recovery Instructions:\n"
            "  1. Inspect the conflicting file(s) listed above to verify unexpected modifications.\n"
            "  2. To explicitly authorize downloading fresh data from Hugging Face Hub and rebuilding "
            "all canonical files, set FORCE_REBUILD = True in Section 1 (Cell 2) and re-run.\n"
            "  3. Alternatively, remove the invalid file(s) from disk and re-run to allow normal missing-file setup."
        )
    else:
        # Files are simply missing (e.g. initial setup)
        missing_list = ", ".join(validation_report["missing_files"])
        print(f"Canonical export files missing: {missing_list}.")
        print("Proceeding with initial dataset acquisition from Hugging Face Hub...")
        print(f"Loading '{DATASET_NAME}' from Hugging Face Hub...")
        dataset = load_dataset(DATASET_NAME)
        print("Dataset successfully loaded from Hugging Face Hub.")
        print(dataset)

✓ All 5 canonical local CSV files exist and passed full chunked content validation.
✓ Skipping Hugging Face Hub dataset load and export operations (Safe Reuse Mode).


,Key,Path,Rows,Columns,Status
0,raw_train,data/raw/train.csv,205734,4,Valid structure & count
1,raw_val,data/raw/validation.csv,1076,4,Valid structure & count
2,raw_test,data/raw/test.csv,1162,4,Valid structure & count
3,processed_train_val,data/processed/train_val.csv,206810,4,Valid structure & count
4,processed_test,data/processed/test.csv,1162,4,Valid structure & count


## 3. Inspect Dataset Splits

Verify the presence of required dataset partitions (`train`, `validation`, and `test`) and evaluate row counts across splits.

In [6]:
EXPECTED_ROW_COUNTS = {
    "train": 205734,
    "validation": 1076,
    "test": 1162,
}
EXPECTED_COLUMNS = {"input", "label", "num_variables", "template"}

if dataset is not None:
    REQUIRED_SPLITS = {"train", "validation", "test"}
    available_splits = set(dataset.keys())

    missing_splits = REQUIRED_SPLITS - available_splits
    if missing_splits:
        raise ValueError(f"Required dataset split(s) missing from loaded dataset: {missing_splits}")

    splits_table = []
    for split_name in ["train", "validation", "test"]:
        split_ds = dataset[split_name]
        splits_table.append({
            "Split": split_name,
            "Record Count": len(split_ds),
            "Features": ", ".join(split_ds.features.keys()),
            "Num Features": len(split_ds.features),
        })

    df_splits_summary = pd.DataFrame(splits_table)
    display(df_splits_summary)

    for split_name, expected_count in EXPECTED_ROW_COUNTS.items():
        actual_count = len(dataset[split_name])
        if actual_count != expected_count:
            raise ValueError(
                f"Row count mismatch for '{split_name}' split: expected {expected_count:,}, got {actual_count:,}"
            )
        actual_cols = set(dataset[split_name].features.keys())
        if actual_cols != EXPECTED_COLUMNS:
            raise ValueError(
                f"Column schema mismatch for '{split_name}' split: expected {EXPECTED_COLUMNS}, got {actual_cols}"
            )

    print("✓ Hugging Face dataset split row counts and schemas explicitly validated.")
else:
    splits_table = []
    for split_name in ["train", "validation", "test"]:
        splits_table.append({
            "Split": split_name,
            "Record Count": EXPECTED_ROW_COUNTS[split_name],
            "Features": ", ".join(sorted(EXPECTED_COLUMNS)),
            "Num Features": len(EXPECTED_COLUMNS),
        })
    df_splits_summary = pd.DataFrame(splits_table)
    display(df_splits_summary)
    print("✓ Canonical local split row counts and schemas verified from preflight validation.")

,Split,Record Count,Features,Num Features
0,train,205734,"input, label, num_variables, template",4
1,validation,1076,"input, label, num_variables, template",4
2,test,1162,"input, label, num_variables, template",4


✓ Canonical local split row counts and schemas verified from preflight validation.


## 4. Inspect Dataset Schema and Samples

Confirm the schema structure from the loaded dataset object, inspect data types, verify missing values and duplicate rows, and inspect sample records.

The expected schema comprises:
- `input`: Formatted premise and hypothesis text.
- `label`: Binary classification label (`0` or `1`).
- `num_variables`: Number of variables involved in the causal system (4, 5, or 6).
- `template`: Semantic causal reasoning relation (e.g., direct cause, collider, confounder, mediator).

In [7]:
if dataset is not None:
    print("Verified Dataset Schema (from Hugging Face):")
    for col_name, feature_def in dataset["train"].features.items():
        print(f"  • {col_name:<15} : {feature_def}")
    df_train = dataset["train"].to_pandas()
    df_val = dataset["validation"].to_pandas()
    df_test = dataset["test"].to_pandas()
else:
    print("Verified Dataset Schema (from Canonical Local CSVs):")
    df_train = pd.read_csv(EXPECTED_EXPORT_PATHS["raw_train"])
    df_val = pd.read_csv(EXPECTED_EXPORT_PATHS["raw_val"])
    df_test = pd.read_csv(EXPECTED_EXPORT_PATHS["raw_test"])
    for col_name, dtype in df_train.dtypes.items():
        print(f"  • {col_name:<15} : {dtype}")

# Check missing values and duplicate records across splits
quality_report = []
for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    quality_report.append({
        "Split": name,
        "Total Rows": len(df),
        "Missing Values": int(df.isnull().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
    })

display(pd.DataFrame(quality_report))

Verified Dataset Schema (from Canonical Local CSVs):


  • input           : str
  • label           : int64
  • num_variables   : int64
  • template        : str


,Split,Total Rows,Missing Values,Duplicate Rows
0,Train,205734,0,0
1,Validation,1076,0,0
2,Test,1162,0,0


In [8]:
print("=" * 80)
print("SAMPLE PREMISE AND HYPOTHESIS PAIR (Train Split, Index 0)")
print("=" * 80)
sample_record = df_train.iloc[0]
print(f"Template      : {sample_record.get('template', 'N/A')}")
print(f"Num Variables : {sample_record.get('num_variables', 'N/A')}")
print(f"Target Label  : {sample_record['label']} ({'Entailment (1)' if sample_record['label'] == 1 else 'Refuted (0)'})")
print(f"\nInput Text:\n{sample_record['input']}")
print("=" * 80)

print("\nSample Records (Top 2 from Train):")
display(df_train.head(2))

SAMPLE PREMISE AND HYPOTHESIS PAIR (Train Split, Index 0)
Template      : has_collider
Num Variables : 4
Target Label  : 0 (Refuted (0))

Input Text:
Premise: Suppose there is a closed system of 4 variables, A, B, C and D. All the statistical relations among these 4 variables are as follows: A correlates with B. A correlates with C. A correlates with D. B correlates with C. B correlates with D. C correlates with D. However, B and D are independent given A. B and D are independent given A and C. C and D are independent given A. C and D are independent given A and B.
Hypothesis: There exists at least one collider (i.e., common effect) of A and B.

Sample Records (Top 2 from Train):


,input,label,num_variables,template
0,Premise: Suppose there is a closed system of 4...,0,4,has_collider
1,Premise: Suppose there is a closed system of 4...,0,4,non-child descendant


## 5. Analyze Label Distribution

Assess target class distribution across splits, as well as the complexity breakdown by variable count and causal template.

In [9]:
label_distribution = []
for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    counts = df["label"].value_counts().to_dict()
    props = df["label"].value_counts(normalize=True).to_dict()
    label_distribution.append({
        "Split": name,
        "Total Rows": len(df),
        "Label 0 (Refuted)": counts.get(0, 0),
        "Label 0 (%)": round(props.get(0, 0.0) * 100, 2),
        "Label 1 (Entailed)": counts.get(1, 0),
        "Label 1 (%)": round(props.get(1, 0.0) * 100, 2),
    })

display(pd.DataFrame(label_distribution))

,Split,Total Rows,Label 0 (Refuted),Label 0 (%),Label 1 (Entailed),Label 1 (%)
0,Train,205734,167686,81.51,38048,18.49
1,Validation,1076,937,87.08,139,12.92
2,Test,1162,982,84.51,180,15.49


In [10]:
if "num_variables" in df_train.columns:
    print("Variable Count Distribution (Train Split):")
    display(df_train["num_variables"].value_counts().sort_index().to_frame("Count"))

if "template" in df_train.columns:
    print("\nTop 10 Causal Templates (Train Split):")
    display(df_train["template"].value_counts().head(10).to_frame("Count"))

Variable Count Distribution (Train Split):


,Count
num_variables,
4,576
5,7524
6,197634



Top 10 Causal Templates (Train Split):


,Count
template,
non-parent ancestor,34356
parent,34292
child,34291
has_collider,34284
has_confounder,34284
non-child descendant,34227


## 6. Export Original Splits

Persist original splits to `data/raw/` (`train.csv`, `validation.csv`, `test.csv`).

### Transactional Staging & Safe Reuse Protocol
Before modifying canonical files, the pipeline inspects all five destinations. In safe reuse mode (`FORCE_REBUILD=False` with pre-validated local files), export operations are safely bypassed without re-writing disk files. If fresh rebuild or initial export is active, all five canonical datasets are staged, pre-validated in staging, and atomically swapped with automated rollback protection.

In [11]:
# Prepare combined dataset reference for preflight destination check
df_combined_ref = pd.concat([df_train, df_val], ignore_index=True)

all_destinations = {
    "raw_train": (EXPECTED_EXPORT_PATHS["raw_train"], df_train, "Raw Train"),
    "raw_val": (EXPECTED_EXPORT_PATHS["raw_val"], df_val, "Raw Validation"),
    "raw_test": (EXPECTED_EXPORT_PATHS["raw_test"], df_test, "Raw Test"),
    "processed_train_val": (EXPECTED_EXPORT_PATHS["processed_train_val"], df_combined_ref, "Processed Combined (Train + Val)"),
    "processed_test": (EXPECTED_EXPORT_PATHS["processed_test"], df_test, "Processed Test"),
}

if dataset is None and not FORCE_REBUILD:
    print("Preflight Destination Inspection (Canonical Local Files):")
    display(pd.DataFrame(validation_report["validation_details"]))
    print("Safe Reuse Mode: Canonical raw datasets already verified on disk; skipping raw export operations.")
    raw_export_reports = [
        {"Dataset": "Raw Train", "Path": str(EXPECTED_EXPORT_PATHS["raw_train"].relative_to(REPO_ROOT)), "Rows": len(df_train), "Action": "Verified and Reused (Pre-validated)"},
        {"Dataset": "Raw Validation", "Path": str(EXPECTED_EXPORT_PATHS["raw_val"].relative_to(REPO_ROOT)), "Rows": len(df_val), "Action": "Verified and Reused (Pre-validated)"},
        {"Dataset": "Raw Test", "Path": str(EXPECTED_EXPORT_PATHS["raw_test"].relative_to(REPO_ROOT)), "Rows": len(df_test), "Action": "Verified and Reused (Pre-validated)"},
    ]
else:
    # Transactional export across all 5 destinations with staging, pre-commit validation, and rollback
    print("Executing transactional export with staging and pre-commit validation...")
    transaction_reports = rebuild_all_canonical_datasets_transactional(all_destinations)
    raw_export_reports = [r for r in transaction_reports if "Raw" in r["Dataset"]]

display(pd.DataFrame(raw_export_reports))

Preflight Destination Inspection (Canonical Local Files):


,Key,Path,Rows,Columns,Status
0,raw_train,data/raw/train.csv,205734,4,Valid structure & count
1,raw_val,data/raw/validation.csv,1076,4,Valid structure & count
2,raw_test,data/raw/test.csv,1162,4,Valid structure & count
3,processed_train_val,data/processed/train_val.csv,206810,4,Valid structure & count
4,processed_test,data/processed/test.csv,1162,4,Valid structure & count


Safe Reuse Mode: Canonical raw datasets already verified on disk; skipping raw export operations.


,Dataset,Path,Rows,Action
0,Raw Train,data/raw/train.csv,205734,Verified and Reused (Pre-validated)
1,Raw Validation,data/raw/validation.csv,1076,Verified and Reused (Pre-validated)
2,Raw Test,data/raw/test.csv,1162,Verified and Reused (Pre-validated)


## 7. Combine Training and Validation

Concatenate the original training (`205,734` rows) and validation (`1,076` rows) splits into a single combined dataset (`206,810` rows).

**Methodological Rationale**:
- **Model Development**: Training and validation sets remain strictly isolated during initial exploration, architecture selection, and hyperparameter tuning so validation performance accurately informs modeling decisions.
- **Final Retraining**: After optimal model architecture and hyperparameters are finalized, retraining on the combined training and validation data maximizes exposure to the full training distribution.
- **Test Set Isolation**: The test set is held out strictly for final evaluation and must never be included in the combined training corpus or used for hyperparameter selection.

In [12]:
if dataset is None and not FORCE_REBUILD:
    df_raw_train_fresh = df_train
    df_raw_val_fresh = df_val
else:
    # Downstream processed files are strictly regenerated from the freshly exported raw splits
    df_raw_train_fresh = pd.read_csv(EXPECTED_EXPORT_PATHS["raw_train"])
    df_raw_val_fresh = pd.read_csv(EXPECTED_EXPORT_PATHS["raw_val"])

df_combined = pd.concat([df_raw_train_fresh, df_raw_val_fresh], ignore_index=True)

# Verify row count equality
expected_combined_count = len(df_raw_train_fresh) + len(df_raw_val_fresh)
assert len(df_combined) == expected_combined_count, (
    f"Combined row count mismatch: expected {expected_combined_count}, got {len(df_combined)}"
)
assert list(df_combined.columns) == list(df_raw_train_fresh.columns), "Column schema mismatch in combined DataFrame"

print(f"Fresh Raw Training rows   : {len(df_raw_train_fresh):,}")
print(f"Fresh Raw Validation rows : {len(df_raw_val_fresh):,}")
print(f"Combined total rows       : {len(df_combined):,}")
print("Row count assertion passed: len(combined) == len(train) + len(val)")

display(df_combined.head(2))

Fresh Raw Training rows   : 205,734
Fresh Raw Validation rows : 1,076
Combined total rows       : 206,810
Row count assertion passed: len(combined) == len(train) + len(val)


,input,label,num_variables,template
0,Premise: Suppose there is a closed system of 4...,0,4,has_collider
1,Premise: Suppose there is a closed system of 4...,0,4,non-child descendant


## 8. Export Combined Dataset

Export the combined training and validation dataset to `data/processed/train_val.csv`.

Additionally, export an exact copy of the test partition to `data/processed/test.csv`. The test partition remains strictly held out for final model evaluation.

In safe reuse mode (`FORCE_REBUILD=False` with verified canonical datasets), export operations are safely skipped because the files already exist and have been validated. In fresh rebuild or initial setup mode, datasets were staged, validated, and transactionally replaced in coordination with raw splits.

In [13]:
if dataset is None and not FORCE_REBUILD:
    print("Safe Reuse Mode: Canonical processed datasets already verified on disk; skipping processed export operations.")
    processed_export_reports = [
        {"Dataset": "Processed Combined (Train + Val)", "Path": str(EXPECTED_EXPORT_PATHS["processed_train_val"].relative_to(REPO_ROOT)), "Rows": len(df_combined), "Action": "Verified and Reused (Pre-validated)"},
        {"Dataset": "Processed Test", "Path": str(EXPECTED_EXPORT_PATHS["processed_test"].relative_to(REPO_ROOT)), "Rows": len(df_test), "Action": "Verified and Reused (Pre-validated)"},
    ]
else:
    # Processed datasets were staged, validated, and transactionally replaced in coordination with raw splits
    processed_export_reports = [r for r in transaction_reports if "Processed" in r["Dataset"]]

display(pd.DataFrame(processed_export_reports))

Safe Reuse Mode: Canonical processed datasets already verified on disk; skipping processed export operations.


,Dataset,Path,Rows,Action
0,Processed Combined (Train + Val),data/processed/train_val.csv,206810,Verified and Reused (Pre-validated)
1,Processed Test,data/processed/test.csv,1162,Verified and Reused (Pre-validated)


## 9. Verify Exported Datasets

Reload each expected CSV file from disk to verify schema fidelity, record count accuracy, serialization consistency, test set isolation, and concatenation order across all columns and rows.

In [14]:
verification_targets = [
    ("Raw Train", EXPECTED_EXPORT_PATHS["raw_train"], df_train),
    ("Raw Validation", EXPECTED_EXPORT_PATHS["raw_val"], df_val),
    ("Raw Test", EXPECTED_EXPORT_PATHS["raw_test"], df_test),
    ("Processed Combined", EXPECTED_EXPORT_PATHS["processed_train_val"], df_combined),
    ("Processed Test", EXPECTED_EXPORT_PATHS["processed_test"], df_test),
]

audit_results = []
for name, file_path, df_source in verification_targets:
    if not file_path.exists():
        raise FileNotFoundError(f"Verification error: Expected file not found: {file_path}")
    
    reloaded_df = pd.read_csv(file_path)
    
    # Verify row count
    if len(reloaded_df) != len(df_source):
        raise ValueError(
            f"Row count mismatch for {name}: expected {len(df_source)}, got {len(reloaded_df)}"
        )
    
    # Verify column structure and ordering
    if list(reloaded_df.columns) != list(df_source.columns):
        raise ValueError(
            f"Column mismatch for {name}: expected {list(df_source.columns)}, got {list(reloaded_df.columns)}"
        )
    
    # Verify all columns and records (accounting for CSV string serialization)
    for col in df_source.columns:
        src_series = df_source[col].fillna("").astype(str).reset_index(drop=True)
        rel_series = reloaded_df[col].fillna("").astype(str).reset_index(drop=True)
        if not src_series.equals(rel_series):
            raise ValueError(f"Value content mismatch in {name} for column '{col}'")
    
    null_count = int(reloaded_df.isnull().sum().sum())
    dup_count = int(reloaded_df.duplicated().sum())
    
    audit_results.append({
        "Dataset": name,
        "File Path": str(file_path.relative_to(REPO_ROOT)),
        "Row Count": len(reloaded_df),
        "Columns": len(reloaded_df.columns),
        "Missing Values": null_count,
        "Duplicate Rows": dup_count,
        "Verified": True,
    })

display(pd.DataFrame(audit_results))

,Dataset,File Path,Row Count,Columns,Missing Values,Duplicate Rows,Verified
0,Raw Train,data/raw/train.csv,205734,4,0,0,True
1,Raw Validation,data/raw/validation.csv,1076,4,0,0,True
2,Raw Test,data/raw/test.csv,1162,4,0,0,True
3,Processed Combined,data/processed/train_val.csv,206810,4,0,0,True
4,Processed Test,data/processed/test.csv,1162,4,0,0,True


### Cross-Split Parity & Partition Alignment Assertions
Verify that raw test identically matches processed test, and processed combined cleanly partitions into train and validation.

In [15]:
# Verify that data/raw/test.csv exactly matches data/processed/test.csv across all columns and records
df_raw_test_check = pd.read_csv(EXPECTED_EXPORT_PATHS["raw_test"])
df_proc_test_check = pd.read_csv(EXPECTED_EXPORT_PATHS["processed_test"])
assert len(df_raw_test_check) == len(df_proc_test_check), "Row count mismatch between raw and processed test"
assert list(df_raw_test_check.columns) == list(df_proc_test_check.columns), "Column mismatch between raw and processed test"
for col in df_raw_test_check.columns:
    raw_s = df_raw_test_check[col].fillna("").astype(str)
    proc_s = df_proc_test_check[col].fillna("").astype(str)
    assert raw_s.equals(proc_s), f"Column '{col}' mismatch between raw test and processed test"

# Verify that data/processed/train_val.csv contains original training records followed by original validation records across all columns
df_proc_combined_check = pd.read_csv(EXPECTED_EXPORT_PATHS["processed_train_val"])
train_len = len(df_train)
val_len = len(df_val)
assert len(df_proc_combined_check) == train_len + val_len, "Combined row count mismatch"
assert list(df_proc_combined_check.columns) == list(df_train.columns), "Combined column mismatch"

# Check all columns and records for training partition (prefix)
df_c_train = df_proc_combined_check.iloc[:train_len].reset_index(drop=True)
for col in df_train.columns:
    expected_s = df_train[col].fillna("").astype(str).reset_index(drop=True)
    actual_s = df_c_train[col].fillna("").astype(str).reset_index(drop=True)
    assert actual_s.equals(expected_s), f"Column '{col}' mismatch in combined training records"

# Check all columns and records for validation partition (suffix)
df_c_val = df_proc_combined_check.iloc[train_len:train_len + val_len].reset_index(drop=True)
for col in df_val.columns:
    expected_s = df_val[col].fillna("").astype(str).reset_index(drop=True)
    actual_s = df_c_val[col].fillna("").astype(str).reset_index(drop=True)
    assert actual_s.equals(expected_s), f"Column '{col}' mismatch in combined validation records"

print("\nIntegrity verification passed: All 5 exported datasets match source data across all columns and records with complete fidelity.")


Integrity verification passed: All 5 exported datasets match source data across all columns and records with complete fidelity.


## 10. Summary

### Dataset Characteristics
- **Benchmark**: Corr2Cause (`causal-nlp/corr2cause`) ingested with verified splits: `train` (205,734 records), `validation` (1,076 records), and `test` (1,162 records).
- **Schema & Data Quality**: Four official features (`input`, `label`, `num_variables`, `template`). All four columns, values, and records are preserved. 0 missing values and 0 duplicate rows detected across all splits.
- **Label Distribution**: Severe class imbalance across all splits (~81.5% Label 0 / refuted causal hypotheses vs. ~18.5% Label 1 / entailed causal hypotheses in train; 87.1% vs. 12.9% in validation; 84.5% vs. 15.5% in test).
- **Target Export Files**:
  - `data/raw/train.csv` (205,734 rows)
  - `data/raw/validation.csv` (1,076 rows)
  - `data/raw/test.csv` (1,162 rows)
  - `data/processed/train_val.csv` (206,810 rows)
  - `data/processed/test.csv` (1,162 rows)
- **Safe Export, Reuse & Transactional Protocol**: The pipeline conducts a preflight full-fidelity integrity check across all five canonical export files. When these files already exist and pass all structural, schema, row-count, and chunked cross-split content consistency checks, Hugging Face Hub dataset loading and file export operations are safely skipped to avoid redundant I/O and network overhead. If any existing canonical file contains corrupted or conflicting data, execution halts immediately with explicit recovery instructions. When `FORCE_REBUILD=True` or during initial setup, datasets are staged in `data/.rebuild_staging`, validated before replacing any canonical data, backed up to `data/.rebuild_backup`, and sequentially swapped via atomic single-file replacements with automated rollback recovery.
- **Evaluation Protocol**: The test set is held out strictly for final integration and model evaluation. It is never utilized for feature engineering, model architecture exploration, or hyperparameter selection.
- **Pipeline Next Step**: With dataset properties and schemas verified, proceed to downstream feature engineering, entity extraction, and modeling pipelines.